# 07 - LLM, Prompt, Chain Và History

Notebook này dùng để **lắp prompt, chain và lịch sử hội thoại**. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.

### BƯỚC 1: Setup Tối Thiểu

- **Tác dụng chính:** Notebook này dùng để lắp prompt, chain và lịch sử hội thoại. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `find_chatbot_fashion_root`, `CHATBOT_FASHION_DIR` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


## BƯỚC 0: Đặt Notebook 07 Đúng Vị Trí Trong Hệ Thống

Notebook 07 **không phân loại intent**. Router ở `app/core/intent.py` đã chọn pipeline trước; các prompt và chain tại đây chỉ biến context đã được chọn thành câu trả lời. Nhờ ranh giới này, việc sửa prompt không thể âm thầm đổi route và việc sửa router không cần nhân bản prompt.


In [ ]:
import sys

from pathlib import Path

import ollama

from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_community.chat_message_histories import RedisChatMessageHistory

from langchain_core.documents import Document

from langchain_core.messages import BaseMessage, SystemMessage

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate

from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_ollama import ChatOllama

def find_chatbot_fashion_root(start: Path | None = None) -> Path:
    """Tìm thư mục dự án có chứa `app/config.py`.

    Args:
        start (Path | None): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        Path: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        for candidate in (parent, parent / "Chatbot_Fashion"):
            if (candidate / "app" / "config.py").exists():
                return candidate
    raise RuntimeError("Không tìm thấy thư mục gốc Chatbot_Fashion chứa app/config.py")


In [ ]:
CHATBOT_FASHION_DIR = find_chatbot_fashion_root()

if str(CHATBOT_FASHION_DIR) not in sys.path:
    sys.path.insert(0, str(CHATBOT_FASHION_DIR))

from app.config import (
    HISTORY_MAX_MESSAGES,
    HISTORY_RECENT_KEEP,
    LLM_MODEL,
    LLM_NUM_CTX,
    LLM_NUM_PREDICT,
    LLM_TEMPERATURE,
    LLM_TIMEOUT,
    OLLAMA_BASE_URL,
    REDIS_URL,
    SUMMARIZE_MAX_TOKENS,
)

print("[OK] Setup notebook 07 hoàn tất")

print(f"Project root: {CHATBOT_FASHION_DIR}")


## Nguyên Tắc Grounding Của Prompt

Product card là nguồn sự thật cho mã, giá, thương hiệu và ảnh. Prompt production chỉ cho LLM giải thích tên/đặc điểm/lý do phù hợp; backend còn lọc các dòng fact thương mại nếu model vẫn tự viết. Notebook import prompt từ `app/core/llm.py` để không tạo một bản prompt thứ hai bị lệch.


### BƯỚC 2: Khởi Tạo LLM Client

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `llm` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=LLM_TEMPERATURE,
    timeout=LLM_TIMEOUT,
    num_predict=LLM_NUM_PREDICT,
    num_ctx=LLM_NUM_CTX,
)

print("[OK] Đã tạo ChatOllama client; chưa gọi model")
print(f"Model       : {LLM_MODEL}")
print(f"Ollama URL  : {OLLAMA_BASE_URL}")
print(f"Temperature : {LLM_TEMPERATURE}")
print(f"Context     : {LLM_NUM_CTX} tokens")
print(f"Output tối đa: {LLM_NUM_PREDICT} tokens")


### BƯỚC 3: Prompt Cho Luồng Tìm Sản Phẩm

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `SEARCH_SYSTEM_PROMPT`, `format_documents_for_llm`, `QA_PROMPT`, `contextualize_q_prompt`, `doc_prompt` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
from app.core.llm import SEARCH_SYSTEM_PROMPT, format_documents_for_llm

print('[OK] Dùng prompt tìm kiếm production; product card là nguồn sự thật')


In [ ]:
from app.core.llm import QA_PROMPT, contextualize_q_prompt, doc_prompt

print('[OK] Dùng QA prompt và document schema production')


### BƯỚC 4: Prompt Cho Luồng Phối Đồ

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `OUTFIT_SYSTEM_PROMPT`, `outfit_prompt` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
from app.core.llm import OUTFIT_SYSTEM_PROMPT

print('[OK] Dùng prompt outfit production')


In [ ]:
from app.core.llm import outfit_prompt

print('[OK] Dùng outfit chain prompt production')


### BƯỚC 5: Redis History Và Tóm Tắt Hội Thoại

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `SUMMARIZE_PROMPT`, `ollama_client` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
from app.core.llm import SUMMARIZE_PROMPT


In [ ]:
ollama_client = ollama.Client(host=OLLAMA_BASE_URL)

print(f"Ngưỡng tóm tắt : trên {HISTORY_MAX_MESSAGES} messages")

print(f"Giữ nguyên      : {HISTORY_RECENT_KEEP} messages gần nhất")

print(f"Token tóm tắt  : tối đa {SUMMARIZE_MAX_TOKENS}")


### BƯỚC 5B: Biến Message Thành Bản Tóm Tắt

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `_message_speaker`, `summarize_history` để các bước sau tiếp tục sử dụng.


In [ ]:
def _message_speaker(message: BaseMessage) -> str:
    """Đổi loại message của LangChain thành nhãn tiếng Việt dễ đọc.

    Args:
        message (BaseMessage): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        str: Kết quả đã xử lý của hàm.
    """
    speaker_by_type = {
        "human": "Khách",
        "ai": "Chatbot",
        "system": "Tóm tắt trước",
    }
    return speaker_by_type.get(message.type, message.type)


def summarize_history(messages: list[BaseMessage]) -> str:
    """Dùng LLM nén các message cũ thành một đoạn tóm tắt ngắn.

    Args:
        messages (list[BaseMessage]): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        str: Kết quả đã xử lý của hàm.
    """
    history_lines = [
        f"{_message_speaker(message)}: {str(message.content)[:300]}"
        for message in messages
    ]
    history_text = "\n".join(history_lines)

    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{
            "role": "user",
            "content": SUMMARIZE_PROMPT.format(history_text=history_text),
        }],
        options={
            "temperature": 0,
            "num_predict": SUMMARIZE_MAX_TOKENS,
        },
    )
    return response["message"]["content"].strip()


### BƯỚC 5C: Đọc Và Nén History Theo session_id

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `get_message_history` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
def get_message_history(session_id: str) -> RedisChatMessageHistory:
    """Lấy lịch sử Redis và tự nén khi số message vượt ngưỡng.

    Args:
        session_id (str): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        RedisChatMessageHistory: Kết quả đã xử lý của hàm.
    """
    history = RedisChatMessageHistory(session_id=session_id, url=REDIS_URL)
    messages = history.messages

    if len(messages) <= HISTORY_MAX_MESSAGES:
        return history

    old_messages = messages[:-HISTORY_RECENT_KEEP]
    recent_messages = messages[-HISTORY_RECENT_KEEP:]
    summary_text = summarize_history(old_messages)

    history.clear()
    history.add_message(SystemMessage(
        content=f"[TÓM TẮT HỘI THOẠI TRƯỚC]: {summary_text}",
    ))
    history.add_messages(recent_messages)
    return history


In [ ]:
print("[OK] Đã khai báo lớp history; chưa kết nối Redis")


### BƯỚC 6: Lắp Ráp Ba Chuỗi Xử Lý

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `retriever`, `history_aware_retriever`, `document_chain`, `rag_chain`, `full_chat_chain` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
print("[THÔNG BÁO] Đang kết nối retriever và lắp ráp các chain...")

# Chain 1: RAG đầy đủ cho tìm kiếm bằng văn bản.
# History-aware retriever dùng LLM để viết lại câu hỏi trước khi search.

from app.core.vector_store import get_product_retriever

retriever = get_product_retriever()
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt,
)

# Stuff documents chain định dạng từng Document bằng `doc_prompt`,
# ghép chúng vào `{context}`, rồi mới gửi prompt hoàn chỉnh cho LLM.
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=QA_PROMPT,
    document_prompt=doc_prompt,
)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

full_chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# Chain 2: image retrieval đã hoàn tất ở bên ngoài. Caller phải dùng
# `format_documents_for_llm()` để tạo chuỗi `{context}` trước khi gọi chain.

product_answer_llm_chain = QA_PROMPT | llm

product_answer_chain_with_history = RunnableWithMessageHistory(
    product_answer_llm_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# Chain 3: Layer A và Layer B đã được ghép thành `{outfit_context}` bên ngoài.

outfit_llm_chain = outfit_prompt | llm

outfit_chain_with_history = RunnableWithMessageHistory(
    outfit_llm_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("[OK] Ba chain đã sẵn sàng")
print(f"QA prompt inputs     : {QA_PROMPT.input_variables}")
print(f"Outfit prompt inputs : {outfit_prompt.input_variables}")
print("Tiếp theo: chạy notebook 08 để xem router chọn chain nào.")
